# Chapter 17 — Preserve Before You Interpret

**Companion to *Applied AI*.**

This notebook accompanies Chapter 17. Every rule an application applies to a
model's response is an **interpretation**, and interpretations turn out to be
wrong.

The chapter's proof is preserved here: the same bytes, read under two
interpreter versions, producing two different call statuses — with no new
provider request.

## Question

**Can yesterday's conclusion change from yesterday's bytes?**

## What this notebook establishes

- The preserved replay table: four cases, each showing the original recorded
  status beside what `v2` derives **from the same observation**.
- The headline case: a call recorded as `succeeded` on day one, reinterpreted
  as `unresolved`, because the provider's own finish reason said `length`.
- A hands-on version of the same move over Chapter 11's real preserved response
  bytes — two interpreters, one observation, zero provider calls.
- What happens when the bytes are gone: the reinterpretation is **refused**, and
  why the dangerous failure is the one that returns a right answer anyway.

## What this notebook does **not** establish

- **`v2` is not "correct".** It is right about this defect class on these
  fixtures. `v3` over the same bytes is the expected future, not a failure.
- **Reinterpretation is not undo.** Anything done under the day-one conclusion
  stays done.
- A matching hash proves the bytes are the ones that were stored. It does not
  prove they are what the provider sent, or that the answer is true.

## Setup

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="reinterpretation"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
REINT = EVIDENCE_DIR / "reinterpretation" / "recorded-call-v1-v2"
table = json.loads((REINT / "replay-table.json").read_text(encoding="utf-8"))

print("bundle:", REINT.name)
print("cases :", len(table))
print("keys  :", list(table[0]))

bundle: recorded-call-v1-v2
cases : 4
keys  : ['case', 'evidence_class', 'extraction_version', 'original_error_kind', 'original_status', 'v1_basis', 'v1_error_kind', 'v1_generation', 'v1_reproduces_original', 'v2_basis', 'v2_basis_detail', 'v2_error_kind', 'v2_generation', 'v2_policy_decision', 'v2_provider_reason']


## 1. Observation, interpretation, decision

```text
observation  !=  interpretation  !=  decision
```

Only the first is a fact about the world. What the bytes **mean** and what to
**do** about them are conclusions — and conclusions are allowed to be wrong,
provided the fact they came from is still there.

In [2]:
LAYERS = [
    ("The provider returned these bytes", "attempt.observed", "never changes"),
    ("What the bytes mean",               "attempt.interpreted",
     "a new version appends a new record; the old one stays"),
    ("What to do about it",               "call.status_decided / call.reinterpreted",
     "appended, never edited; the adopted view is the latest"),
    ("What happens next",                 "work-state projection, acceptance",
     "re-derived from the adopted record"),
]
for what, where, note in LAYERS:
    print(f"  {what:<36}{where:<44}{note}")

  The provider returned these bytes   attempt.observed                            never changes
  What the bytes mean                 attempt.interpreted                         a new version appends a new record; the old one stays
  What to do about it                 call.status_decided / call.reinterpreted    appended, never edited; the adopted view is the latest
  What happens next                   work-state projection, acceptance           re-derived from the adopted record


## 2. The preserved replay table

Four cases. Runs 1–3 preserve no provider payload — the only surviving evidence
is inside CodeAI-authored error strings. Run 4 preserves a decoded provider
payload. The bundle records that distinction as `evidence_class`, and it limits
what `v2` is allowed to conclude.

In [3]:
print(f"{'case':<20}{'evidence_class':<28}{'original':<12}{'v1 reproduces it?'}")
print("-" * 82)
for r in table:
    print(f"{r['case']:<20}{r['evidence_class']:<28}"
          f"{r['original_status']:<12}{r['v1_reproduces_original']}")

assert all(r["v1_reproduces_original"] for r in table)
print()
print("assertion held: v1 reproduces all four historical labels (4/4).")
print()
print("That control matters. If the old reading could not be reproduced, we")
print("could not tell a corrected interpretation from a broken replay.")

case                evidence_class              original    v1 reproduces it?
----------------------------------------------------------------------------------
run4-success        decoded_provider_payload    succeeded   True
1-edge-rejection    adapter_error_text          failed      True
2-wrong-protocol    adapter_error_text          failed      True
3-missing-session   adapter_error_text          failed      True

assertion held: v1 reproduces all four historical labels (4/4).

That control matters. If the old reading could not be reproduced, we
could not tell a corrected interpretation from a broken replay.


## What v2 changed — and what it refused to change

In [4]:
print(f"{'case':<20}{'v1 generation':<16}{'v2 generation':<16}"
      f"{'v1 error kind':<24}{'v2 error kind'}")
print("-" * 100)
for r in table:
    print(f"{r['case']:<20}{str(r['v1_generation']):<16}{str(r['v2_generation']):<16}"
          f"{str(r['v1_error_kind']):<24}{str(r['v2_error_kind'])}")

print()
for r in table:
    changed = (r["v1_generation"] != r["v2_generation"]
               or r["v1_error_kind"] != r["v2_error_kind"])
    if changed:
        detail = r.get("v2_basis_detail") or {}
        sig = detail.get("signature") or r.get("v2_provider_reason")
        print(f"CHANGED  {r['case']:<20} basis={r['v2_basis']!r} evidence={sig!r}")
    else:
        print(f"unchanged {r['case']:<20} v2 found no evidence to justify a new label")

case                v1 generation   v2 generation   v1 error kind           v2 error kind
----------------------------------------------------------------------------------------------------
run4-success        complete        truncated       None                    None
1-edge-rejection    unknown         unknown         authentication_error    edge_rejected
2-wrong-protocol    unknown         unknown         provider_error          provider_error
3-missing-session   unknown         unknown         provider_error          invalid_request

CHANGED  run4-success         basis=None evidence='length'
CHANGED  1-edge-rejection     basis='body_signature' evidence='cloudflare_1010_v1'
unchanged 2-wrong-protocol     v2 found no evidence to justify a new label
CHANGED  3-missing-session    basis='body_signature' evidence='missing_session_id_v1'


## Observation

Three of the four moved, and each move names the evidence that licensed it:

- **run4-success** — generation `complete` becomes `truncated`, on the
  provider's own reason `length`, quoted verbatim.
- **1-edge-rejection** — `authentication_error` becomes `edge_rejected`,
  because a Cloudflare signature was matched **in the preserved text**, with the
  source recorded.
- **3-missing-session** — reclassified on a matched signature in the same way.

The fourth is the important one.

In [5]:
r = next(x for x in table if x["case"] == "2-wrong-protocol")
print("2-wrong-protocol:")
print("   v1 error kind :", r["v1_error_kind"])
print("   v2 error kind :", r["v2_error_kind"])
print("   v2 basis      :", r["v2_basis"])
print()
print("The chapter's prose diagnoses this run as the wrong dialect for the")
print("model. v2 does NOT adopt that diagnosis: no `wrong_protocol` label")
print("exists in the classifier, and the HTTP 500 body carries no signature")
print("that would license one. So the label stays `provider_error`.")
print()
print("v2 refuses to invent causes. A human diagnosis in the manuscript does")
print("not feed back into classification - which is what keeps the replay a")
print("reinterpretation rather than a re-labelling.")

2-wrong-protocol:
   v1 error kind : provider_error
   v2 error kind : provider_error
   v2 basis      : adapter_reported

The chapter's prose diagnoses this run as the wrong dialect for the
model. v2 does NOT adopt that diagnosis: no `wrong_protocol` label
exists in the classifier, and the HTTP 500 body carries no signature
that would license one. So the label stays `provider_error`.

v2 refuses to invent causes. A human diagnosis in the manuscript does
not feed back into classification - which is what keeps the replay a
reinterpretation rather than a re-labelling.


The bundle is explicit about a further limit, and it is worth carrying: the
ch11 artifacts are **legacy envelopes, not pure observations**. Runs 1–3
preserve no provider payload at all — one of the error strings is itself
truncated. That is exactly the weakness this chapter exists to remove, caught
in the book's own earlier evidence.

## 3. The same move, by hand, on Chapter 11's real bytes

The preserved live response from Chapter 11 is in this repository. Two
interpreters, one observation, **no provider call**.

In [6]:
LIVE = EVIDENCE_DIR / "ch11-live-opencode" / "artifacts"
obs_path = sorted(LIVE.glob("*.json"))[0]
obs = json.loads(obs_path.read_text(encoding="utf-8"))

import hashlib
raw_bytes = obs_path.read_bytes()
digest = hashlib.sha256(raw_bytes).hexdigest()
print("observation file :", obs_path.name)
print("sha256 of bytes  :", digest[:24], "...")
print("stored under name:", obs_path.stem[:24], "...")

observation file : 1a459bea7ba6b4726f7387cbb0fa8482e66786b82aa2b2799276f4a1127bff05.json
sha256 of bytes  : 1a459bea7ba6b4726f7387cb ...
stored under name: 1a459bea7ba6b4726f7387cb ...


In [7]:
COMPLETION_MAP = {
    ("chat_completions", "stop"):       "complete",
    ("chat_completions", "length"):     "truncated",
    ("messages", "end_turn"):           "complete",
    ("messages", "max_tokens"):         "truncated",
    ("responses", "completed"):         "complete",
    ("responses", "max_output_tokens"): "truncated",
}

def interpret_v1(observation):
    """The historical rule: text means complete."""
    text = observation.get("output_text") or ""
    return {"version": "v1", "generation": "complete" if text else "empty",
            "basis": "output text present"}

def interpret_v2(observation):
    """The corrected rule: the provider's own reason decides."""
    protocol = observation.get("protocol")
    pr = observation.get("provider_response", {})
    reason = None
    if "choices" in pr:
        reason = pr["choices"][0].get("finish_reason")
    state = COMPLETION_MAP.get((protocol, reason), "unknown")
    return {"version": "v2", "generation": state,
            "basis": f"provider reason {reason!r}"}

v1 = interpret_v1(obs)
v2 = interpret_v2(obs)
for v in (v1, v2):
    print(f"  {v['version']}: generation={v['generation']:<12} basis={v['basis']}")

  v1: generation=complete     basis=output text present
  v2: generation=truncated    basis=provider reason 'length'


In [8]:
def decide(interpretation):
    """Policy over an interpretation, not over the bytes."""
    g = interpretation["generation"]
    if g == "complete":
        return "succeeded", "final attempt accepted"
    if g == "truncated":
        return "unresolved", "generation truncated, not treated as completed cognition"
    if g == "empty":
        return "failed", "no output text"
    return "succeeded", "no error recorded"

s1, r1 = decide(v1)
s2, r2 = decide(v2)

print(f"day one  (v1): call status = {s1:<12} ({r1})")
print(f"day two  (v2): call status = {s2:<12} ({r2})")
print()
print("provider calls made by this cell: 0")
print("bytes on disk:", "unchanged" if hashlib.sha256(obs_path.read_bytes()).hexdigest() == digest else "CHANGED")

assert s1 == "succeeded" and s2 == "unresolved"
print()
print("assertion held: succeeded -> unresolved, from the same observation")

day one  (v1): call status = succeeded    (final attempt accepted)
day two  (v2): call status = unresolved   (generation truncated, not treated as completed cognition)

provider calls made by this cell: 0
bytes on disk: unchanged

assertion held: succeeded -> unresolved, from the same observation


## Interpretation

Nothing was asked of the model. What changed is what the system **concludes**
from what it already had — and therefore what it would do next.

In [9]:
NEXT = {
    "succeeded":  "check and accept",
    "unresolved": "start the call",
}
print(f"under v1 the task's next step is: {NEXT[s1]!r}")
print(f"under v2 the task's next step is: {NEXT[s2]!r}")
print()
print("And an acceptance citing the day-one interpretation is now refused")
print("for two reasons the chapter names:")
print("   source_call_not_succeeded        (the adopted status is unresolved)")
print("   interpretation_not_decision_basis (the cited reading is no longer")
print("                                      the one the status rests on)")

under v1 the task's next step is: 'check and accept'
under v2 the task's next step is: 'start the call'

And an acceptance citing the day-one interpretation is now refused
for two reasons the chapter names:
   source_call_not_succeeded        (the adopted status is unresolved)
   interpretation_not_decision_basis (the cited reading is no longer
                                      the one the status rests on)


## 4. Without the bytes

The chapter damaged two copies: one with the response file deleted, one with
its bytes altered and the recorded hash left alone.

In [10]:
class ObservationUnavailable(Exception):
    pass

def reinterpret(stored_bytes, recorded_sha, interpreter):
    if stored_bytes is None:
        raise ObservationUnavailable("response body bytes are missing")
    if hashlib.sha256(stored_bytes).hexdigest() != recorded_sha:
        raise ObservationUnavailable("bytes do not match their recorded sha256")
    return interpreter(json.loads(stored_bytes.decode("utf-8")))

print("intact  :", reinterpret(raw_bytes, digest, interpret_v2)["generation"])

for label, payload in (("deleted", None),
                       ("corrupted", raw_bytes.replace(b"length", b"stop  "))):
    try:
        reinterpret(payload, digest, interpret_v2)
    except ObservationUnavailable as exc:
        print(f"{label:<9}: REFUSED - {exc}")

print()
print("The runtime can still tell you that a response arrived, when, and what")
print("hash it had. It can no longer tell you what the response MEANT under a")
print("rule written after the fact.")

intact  : truncated
deleted  : REFUSED - response body bytes are missing
corrupted: REFUSED - bytes do not match their recorded sha256

The runtime can still tell you that a response arrived, when, and what
hash it had. It can no longer tell you what the response MEANT under a
rule written after the fact.


## 5. The dangerous failure

Before the fix, CodeAI's projection did not refuse when the bytes were gone. It
fell back to a **derived copy** made at execution time — and then attached the
observation's artifact reference to the result anyway.

In [11]:
derived_copy = {"protocol": obs.get("protocol"),
                "provider_response": obs.get("provider_response"),
                "output_text": obs.get("output_text")}

def reinterpret_old(stored_bytes, recorded_sha, derived, interpreter):
    """The behaviour before the fix: silently substitute, keep the name."""
    if stored_bytes is None:
        result = interpreter(derived)
        result["evidence"] = recorded_sha        # names bytes that are not there
        result["substituted"] = True
        return result
    return interpreter(json.loads(stored_bytes.decode("utf-8")))

old = reinterpret_old(None, digest, derived_copy, interpret_v2)
print("old behaviour with the bytes deleted:")
print("   generation :", old["generation"], " <- happens to be RIGHT")
print("   evidence   :", old["evidence"][:24], "...")
print("   bytes with that hash on disk? ", False)
print()
print("That is the dangerous version of this failure.")
print()
print("A wrong answer gets investigated. A RIGHT answer that names evidence")
print("which is not there passes review, and stays wrong about its provenance")
print("for as long as anyone relies on it.")

old behaviour with the bytes deleted:
   generation : truncated  <- happens to be RIGHT
   evidence   : 1a459bea7ba6b4726f7387cb ...
   bytes with that hash on disk?  False

That is the dangerous version of this failure.

A wrong answer gets investigated. A RIGHT answer that names evidence
which is not there passes review, and stays wrong about its provenance
for as long as anyone relies on it.


## Interpretation

Five rules come out of this chapter, and the notebook has exercised four.

1. **Preserve once, interpret many times.** Store the response bytes,
   content-addressed, before extracting anything.
2. **Reinterpret; do not re-ask.** A corrected rule applied to preserved bytes
   answers *what you actually received*. Asking again answers a different
   question, costs money again, and may come back different.
3. **Append; never edit.** A new reading adds a record and becomes the adopted
   one. The old reading, and the decisions made under it, stay in history.
4. **No bytes, no reading.** Refuse, rather than substituting a derived copy
   under the original's name.
5. **Reinterpretation is not undo.** It changes what the system does next, not
   what it already did. (Not exercised here — it needs the effect machinery of
   Chapters 19 to 22.)

## Try it yourself

1. **Write v3.** Add `content_filter -> filtered` and an explicit `refusal`
   state, then re-read the same bytes. Which decisions would you leave alone,
   and why does that matter more than getting v3 right?
2. **Make v2 wrong.** Map `length` to `complete` and re-run. The pipeline is
   perfectly happy. What in the record would let a later reader catch you?
3. **Break the hash check.** Remove the digest comparison from `reinterpret`
   and feed it the corrupted bytes. You now have a confident reading of a
   response nobody sent.
4. **Version your own parser.** Find one rule your code applies to model
   responses. Could you re-run a corrected version over last month's calls
   without calling the model again? If not, that is the gap this chapter is
   about.